# Polars e análise de dados — guia aprofundado com Pandas

**Trilha complementar | Nível:** iniciante → intermediário | **Formato:** teoria, demonstração, prática e consulta rápida.

> **Objetivo:** aprender a pensar em operações de dados com Polars, compreender *expressions* e processamento *lazy*, e reconhecer diferenças em relação ao Pandas. O material é autocontido: todos os dados são fictícios e criados no notebook.

**Pré-requisitos:** Python básico, noções de tabelas; experiência com Pandas ajuda, mas não é obrigatória.

**Roteiro:** 1. Ambiente; 2. Modelo mental; 3. DataFrames; 4. Seleção e expressões; 5. Limpeza e tipos; 6. Agrupamentos e joins; 7. Datas, janelas e reorganização; 8. Entrada/saída; 9. Lazy API; 10. Comparação com Pandas; 11. Projeto aplicado; 12. Exercícios resolvidos; 13. Consulta Rápida.

**Como estudar:** execute as células em ordem. Leia o que cada resultado significa, altere valores e refaça os desafios antes de ver as soluções. Não há downloads de datasets externos.

### Materiais complementares

- [Material de referência — Polars e análise de dados](./reference_materials/polars_e_analise_de_dados.pdf)


## 1. Preparação do ambiente

No terminal do ambiente Python selecionado pelo JupyterLab:

```bash
python -m pip install polars pandas ipykernel
```

Este comando é apenas uma referência: **não o execute indiscriminadamente no ambiente compartilhado**. Confira as versões existentes e o `requirements.txt` da disciplina antes de instalar. Se o kernel estiver aberto durante uma instalação necessária, reinicie-o. O notebook não tenta instalar bibliotecas automaticamente. A instalação de `pandas` serve apenas para as comparações; não é requisito do Polars em si. Não é necessário instalar NumPy para estas demonstrações.

In [1]:
from pathlib import Path
from datetime import date
import polars as pl
import pandas as pd

print("Polars:", pl.__version__)
print("Pandas:", pd.__version__)

# Caminho relativo à pasta em que o notebook está sendo executado.
SAIDAS = Path("outputs")
SAIDAS.mkdir(exist_ok=True)
print("Saídas:", SAIDAS.resolve())

Polars: 2.1.0
Pandas: 3.1.1
Saídas: /run/media/eduardo/Backup/Faculdades/PUC - CIENCIA DE DADOS/PUC/02_Disciplinas/00_Recursos_Complementares/outputs


### 1.1 Base fictícia reutilizada ao longo do notebook

Cada linha é um item vendido em uma transação simulada. Valores monetários são didáticos, não contábeis. `preco_unitario` e `quantidade` permitem calcular `receita`. `desconto_pct` contém uma ausência intencional, usada no módulo de limpeza.

**Boa prática:** construir dados de exemplo determinísticos facilita validar os resultados em qualquer computador.

In [2]:
vendas = pl.DataFrame({
    "pedido_id": [101, 102, 103, 104, 105, 106, 107, 108],
    "data": [date(2026, 1, 3), date(2026, 1, 3), date(2026, 1, 4), date(2026, 1, 5),
             date(2026, 2, 1), date(2026, 2, 2), date(2026, 2, 2), date(2026, 2, 3)],
    "produto": ["Caderno", "Caneta", "Mochila", "Caderno", "Caneta", "Mochila", "Caderno", "Caneta"],
    "categoria": ["Papelaria", "Papelaria", "Acessórios", "Papelaria", "Papelaria", "Acessórios", "Papelaria", "Papelaria"],
    "regiao": ["Sul", "Norte", "Sul", "Norte", "Sul", "Norte", "Sul", "Norte"],
    "quantidade": [2, 5, 1, 3, 4, 2, 1, 6],
    "preco_unitario": [25.0, 4.0, 150.0, 25.0, 4.0, 150.0, 25.0, 4.0],
    "desconto_pct": [0.0, 0.0, 0.10, None, 0.0, 0.05, 0.0, 0.0],
})
print(vendas)
assert vendas.shape == (8, 8)

shape: (8, 8)
┌───────────┬────────────┬─────────┬────────────┬────────┬────────────┬──────────────┬─────────────┐
│ pedido_id ┆ data       ┆ produto ┆ categoria  ┆ regiao ┆ quantidade ┆ preco_unitar ┆ desconto_pc │
│ ---       ┆ ---        ┆ ---     ┆ ---        ┆ ---    ┆ ---        ┆ io           ┆ t           │
│ i64       ┆ date       ┆ str     ┆ str        ┆ str    ┆ i64        ┆ ---          ┆ ---         │
│           ┆            ┆         ┆            ┆        ┆            ┆ f64          ┆ f64         │
╞═══════════╪════════════╪═════════╪════════════╪════════╪════════════╪══════════════╪═════════════╡
│ 101       ┆ 2026-01-03 ┆ Caderno ┆ Papelaria  ┆ Sul    ┆ 2          ┆ 25.0         ┆ 0.0         │
│ 102       ┆ 2026-01-03 ┆ Caneta  ┆ Papelaria  ┆ Norte  ┆ 5          ┆ 4.0          ┆ 0.0         │
│ 103       ┆ 2026-01-04 ┆ Mochila ┆ Acessórios ┆ Sul    ┆ 1          ┆ 150.0        ┆ 0.1         │
│ 104       ┆ 2026-01-05 ┆ Caderno ┆ Papelaria  ┆ Norte  ┆ 3          ┆ 25.0 

## 2. Modelo mental: o que muda em relação ao Pandas?

- **Series:** uma coluna com tipo homogêneo; **DataFrame:** tabela com colunas tipadas.
- **Sem índice de rótulos:** não existe o índice de linhas típico do Pandas. Trabalhamos com colunas explícitas e filtros.
- **Expressões:** `pl.col("quantidade") * 2` descreve uma operação; o contexto (`select`, `with_columns`, `filter`, `agg`) determina como ela é aplicada.
- **Eager:** resultado calculado imediatamente. **Lazy:** plano de consulta montado e executado apenas com `collect()`.
- **Tipos e valores ausentes:** `null` e `NaN` não são a mesma coisa; conversões costumam ser mais explícitas.

**Evite pensar em “traduzir Pandas linha por linha”.** O ganho pedagógico do Polars está em aprender a construir consultas com expressões.

## 3. Series, DataFrame e inspeção inicial

Uma `Series` corresponde a uma coluna. Em um `DataFrame`, `.schema` mostra o nome e o tipo de cada coluna. `.head()`, `.shape`, `.columns` e `.describe()` ajudam na exploração.

In [3]:
serie = pl.Series("idades", [18, 24, 31, 42])
print(serie)
print("Tipo:", serie.dtype)
print("Dimensões:", vendas.shape)
print("Colunas:", vendas.columns)
print("Esquema:", vendas.schema)
print(vendas.head(3))
print(vendas.describe())

shape: (4,)
Series: 'idades' [i64]
[
	18
	24
	31
	42
]
Tipo: Int64
Dimensões: (8, 8)
Colunas: ['pedido_id', 'data', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario', 'desconto_pct']
Esquema: Schema({'pedido_id': Int64, 'data': Date, 'produto': String, 'categoria': String, 'regiao': String, 'quantidade': Int64, 'preco_unitario': Float64, 'desconto_pct': Float64})
shape: (3, 8)
┌───────────┬────────────┬─────────┬────────────┬────────┬────────────┬──────────────┬─────────────┐
│ pedido_id ┆ data       ┆ produto ┆ categoria  ┆ regiao ┆ quantidade ┆ preco_unitar ┆ desconto_pc │
│ ---       ┆ ---        ┆ ---     ┆ ---        ┆ ---    ┆ ---        ┆ io           ┆ t           │
│ i64       ┆ date       ┆ str     ┆ str        ┆ str    ┆ i64        ┆ ---          ┆ ---         │
│           ┆            ┆         ┆            ┆        ┆            ┆ f64          ┆ f64         │
╞═══════════╪════════════╪═════════╪════════════╪════════╪════════════╪══════════════╪═════════════╡

### 3.1 Tipos explícitos

Quando a inferência não atende à regra de negócio, use `cast`. Em dados reais, valide o significado da coluna antes de converter.

In [4]:
exemplo_tipos = pl.DataFrame({"codigo": ["1", "2", "3"], "ativo": [True, False, True]})
convertido = exemplo_tipos.with_columns(pl.col("codigo").cast(pl.Int64))
print(exemplo_tipos.schema)
print(convertido.schema)
assert convertido["codigo"].dtype == pl.Int64

Schema({'codigo': String, 'ativo': Boolean})
Schema({'codigo': Int64, 'ativo': Boolean})


## 4. O coração do Polars: expressões

- `select(...)` escolhe e calcula as colunas do resultado.
- `with_columns(...)` preserva as demais colunas, adicionando ou substituindo colunas nomeadas.
- `filter(...)` conserva apenas as linhas que satisfazem a condição.
- `alias(...)` nomeia o resultado de uma expressão.
- Use parênteses em condições compostas e `&`/`|`, em vez de `and`/`or` em expressões de colunas.

In [5]:
selecao = vendas.select("pedido_id", "produto", "quantidade")
filtradas = vendas.filter((pl.col("quantidade") >= 2) & (pl.col("regiao") == "Sul"))
com_receita_bruta = vendas.with_columns(
    (pl.col("quantidade") * pl.col("preco_unitario")).alias("receita_bruta")
)
print("Seleção:\n", selecao)
print("Filtro:\n", filtradas)
print("Coluna calculada:\n", com_receita_bruta.head(3))

Seleção:
 shape: (8, 3)
┌───────────┬─────────┬────────────┐
│ pedido_id ┆ produto ┆ quantidade │
│ ---       ┆ ---     ┆ ---        │
│ i64       ┆ str     ┆ i64        │
╞═══════════╪═════════╪════════════╡
│ 101       ┆ Caderno ┆ 2          │
│ 102       ┆ Caneta  ┆ 5          │
│ 103       ┆ Mochila ┆ 1          │
│ 104       ┆ Caderno ┆ 3          │
│ 105       ┆ Caneta  ┆ 4          │
│ 106       ┆ Mochila ┆ 2          │
│ 107       ┆ Caderno ┆ 1          │
│ 108       ┆ Caneta  ┆ 6          │
└───────────┴─────────┴────────────┘
Filtro:
 shape: (2, 8)
┌───────────┬────────────┬─────────┬───────────┬────────┬────────────┬──────────────┬──────────────┐
│ pedido_id ┆ data       ┆ produto ┆ categoria ┆ regiao ┆ quantidade ┆ preco_unitar ┆ desconto_pct │
│ ---       ┆ ---        ┆ ---     ┆ ---       ┆ ---    ┆ ---        ┆ io           ┆ ---          │
│ i64       ┆ date       ┆ str     ┆ str       ┆ str    ┆ i64        ┆ ---          ┆ f64          │
│           ┆            ┆     

### 4.1 Encadeamento de expressões e derivação de métricas

Para calcular receita líquida, tratamos desconto ausente como zero **por decisão explícita deste exercício**, não como regra universal. Em dados reais, uma ausência pode indicar problema a investigar.

In [6]:
vendas_calculadas = vendas.with_columns(
    receita_bruta=(pl.col("quantidade") * pl.col("preco_unitario")),
    desconto_tratado=pl.col("desconto_pct").fill_null(0.0),
).with_columns(
    receita_liquida=(pl.col("receita_bruta") * (1 - pl.col("desconto_tratado"))).round(2)
)
print(vendas_calculadas.select("pedido_id", "receita_bruta", "desconto_tratado", "receita_liquida"))
assert vendas_calculadas.filter(pl.col("pedido_id") == 103).item(0, "receita_liquida") == 135.0

shape: (8, 4)
┌───────────┬───────────────┬──────────────────┬─────────────────┐
│ pedido_id ┆ receita_bruta ┆ desconto_tratado ┆ receita_liquida │
│ ---       ┆ ---           ┆ ---              ┆ ---             │
│ i64       ┆ f64           ┆ f64              ┆ f64             │
╞═══════════╪═══════════════╪══════════════════╪═════════════════╡
│ 101       ┆ 50.0          ┆ 0.0              ┆ 50.0            │
│ 102       ┆ 20.0          ┆ 0.0              ┆ 20.0            │
│ 103       ┆ 150.0         ┆ 0.1              ┆ 135.0           │
│ 104       ┆ 75.0          ┆ 0.0              ┆ 75.0            │
│ 105       ┆ 16.0          ┆ 0.0              ┆ 16.0            │
│ 106       ┆ 300.0         ┆ 0.05             ┆ 285.0           │
│ 107       ┆ 25.0          ┆ 0.0              ┆ 25.0            │
│ 108       ┆ 24.0          ┆ 0.0              ┆ 24.0            │
└───────────┴───────────────┴──────────────────┴─────────────────┘


### 4.2 Condicionais `when` / `then` / `otherwise`

Cada ramo precisa ser uma expressão válida: o Polars pode avaliar ramos em paralelo. `pl.lit(...)` marca valores literais quando necessário.

In [7]:
faixas = vendas_calculadas.with_columns(
    faixa_pedido=pl.when(pl.col("receita_liquida") >= 100)
    .then(pl.lit("Alto"))
    .otherwise(pl.lit("Baixo"))
)
print(faixas.select("pedido_id", "receita_liquida", "faixa_pedido"))

shape: (8, 3)
┌───────────┬─────────────────┬──────────────┐
│ pedido_id ┆ receita_liquida ┆ faixa_pedido │
│ ---       ┆ ---             ┆ ---          │
│ i64       ┆ f64             ┆ str          │
╞═══════════╪═════════════════╪══════════════╡
│ 101       ┆ 50.0            ┆ Baixo        │
│ 102       ┆ 20.0            ┆ Baixo        │
│ 103       ┆ 135.0           ┆ Alto         │
│ 104       ┆ 75.0            ┆ Baixo        │
│ 105       ┆ 16.0            ┆ Baixo        │
│ 106       ┆ 285.0           ┆ Alto         │
│ 107       ┆ 25.0            ┆ Baixo        │
│ 108       ┆ 24.0            ┆ Baixo        │
└───────────┴─────────────────┴──────────────┘


## 5. Limpeza e qualidade de dados

As operações de limpeza só fazem sentido quando acompanhadas de critérios de negócio. Diferencie valores ausentes, duplicações, textos inconsistentes e tipos inadequados.

In [8]:
dados_sujos = pl.DataFrame({
    "cliente": [" Ana ", "bruno", "ANA", "Carla", "Carla"],
    "idade_texto": ["28", "34", "?", "41", "41"],
    "nota": [8.0, None, float("nan"), 7.5, 7.5],
})
print("Nulos por coluna:\n", dados_sujos.null_count())
print("Há NaN na coluna nota:\n", dados_sujos.select(pl.col("nota").is_nan().sum()))

limpos = dados_sujos.with_columns(
    cliente=pl.col("cliente").str.strip_chars().str.to_titlecase(),
    idade=pl.col("idade_texto").cast(pl.Int64, strict=False),
    nota=pl.col("nota").fill_nan(None),
).drop("idade_texto").unique(maintain_order=True)
print(limpos)
print("Nulos após limpeza:\n", limpos.null_count())

Nulos por coluna:
 shape: (1, 3)
┌─────────┬─────────────┬──────┐
│ cliente ┆ idade_texto ┆ nota │
│ ---     ┆ ---         ┆ ---  │
│ u32     ┆ u32         ┆ u32  │
╞═════════╪═════════════╪══════╡
│ 0       ┆ 0           ┆ 1    │
└─────────┴─────────────┴──────┘
Há NaN na coluna nota:
 shape: (1, 1)
┌──────┐
│ nota │
│ ---  │
│ u32  │
╞══════╡
│ 1    │
└──────┘
shape: (4, 3)
┌─────────┬──────┬───────┐
│ cliente ┆ nota ┆ idade │
│ ---     ┆ ---  ┆ ---   │
│ str     ┆ f64  ┆ i64   │
╞═════════╪══════╪═══════╡
│ Ana     ┆ 8.0  ┆ 28    │
│ Bruno   ┆ null ┆ 34    │
│ Ana     ┆ null ┆ null  │
│ Carla   ┆ 7.5  ┆ 41    │
└─────────┴──────┴───────┘
Nulos após limpeza:
 shape: (1, 3)
┌─────────┬──────┬───────┐
│ cliente ┆ nota ┆ idade │
│ ---     ┆ ---  ┆ ---   │
│ u32     ┆ u32  ┆ u32   │
╞═════════╪══════╪═══════╡
│ 0       ┆ 2    ┆ 1     │
└─────────┴──────┴───────┘


**Interpretação:** `strict=False` transforma valores que não conseguem ser convertidos em `null`; isso evita uma falha imediata, mas exige inspecionar quantos valores foram perdidos. `fill_nan(None)` converte `NaN` em nulo na coluna de ponto flutuante. `unique()` remove linhas idênticas; não confunda repetição legítima com duplicata indevida.

### 5.1 Texto, ordenação, renomeação e seleção por tipo

Consulte nomes e tipos do esquema ao preparar dados recebidos de fontes externas.

In [9]:
texto = pl.DataFrame({"cidade": ["são paulo", " RECIFE ", "Belo Horizonte"]})
texto_padronizado = texto.with_columns(
    pl.col("cidade").str.strip_chars().str.to_titlecase().alias("cidade_padronizada")
)
ordenadas = vendas.sort(["regiao", "quantidade"], descending=[False, True])
renomeadas = vendas.rename({"regiao": "regiao_venda"})
print(texto_padronizado)
print(ordenadas.select("regiao", "quantidade").head(4))
print(renomeadas.columns)

shape: (3, 2)
┌────────────────┬────────────────────┐
│ cidade         ┆ cidade_padronizada │
│ ---            ┆ ---                │
│ str            ┆ str                │
╞════════════════╪════════════════════╡
│ são paulo      ┆ São Paulo          │
│  RECIFE        ┆ Recife             │
│ Belo Horizonte ┆ Belo Horizonte     │
└────────────────┴────────────────────┘
shape: (4, 2)
┌────────┬────────────┐
│ regiao ┆ quantidade │
│ ---    ┆ ---        │
│ str    ┆ i64        │
╞════════╪════════════╡
│ Norte  ┆ 6          │
│ Norte  ┆ 5          │
│ Norte  ┆ 3          │
│ Norte  ┆ 2          │
└────────┴────────────┘
['pedido_id', 'data', 'produto', 'categoria', 'regiao_venda', 'quantidade', 'preco_unitario', 'desconto_pct']


## 6. Agrupamentos e tabelas relacionadas

`group_by(...).agg(...)` calcula métricas por grupo. A ordem das linhas após um agrupamento não deve ser presumida; use `sort` para resultados reproduzíveis.

In [10]:
resumo_regiao = (
    vendas_calculadas.group_by("regiao")
    .agg(
        quantidade_total=pl.col("quantidade").sum(),
        receita_total=pl.col("receita_liquida").sum().round(2),
        pedidos=pl.len(),
        ticket_medio_item=pl.col("receita_liquida").mean().round(2),
    )
    .sort("regiao")
)
print(resumo_regiao)
assert resumo_regiao["pedidos"].sum() == vendas.height

shape: (2, 5)
┌────────┬──────────────────┬───────────────┬─────────┬───────────────────┐
│ regiao ┆ quantidade_total ┆ receita_total ┆ pedidos ┆ ticket_medio_item │
│ ---    ┆ ---              ┆ ---           ┆ ---     ┆ ---               │
│ str    ┆ i64              ┆ f64           ┆ u32     ┆ f64               │
╞════════╪══════════════════╪═══════════════╪═════════╪═══════════════════╡
│ Norte  ┆ 16               ┆ 404.0         ┆ 4       ┆ 101.0             │
│ Sul    ┆ 8                ┆ 226.0         ┆ 4       ┆ 56.5              │
└────────┴──────────────────┴───────────────┴─────────┴───────────────────┘


### 6.1 `join`: relacionamento entre tabelas

Conecte transações a um cadastro auxiliar. **Chaves duplicadas podem multiplicar linhas**; por isso valide a cardinalidade antes da junção.

In [11]:
cadastro = pl.DataFrame({
    "produto": ["Caderno", "Caneta", "Mochila"],
    "fornecedor": ["Fornecedor A", "Fornecedor B", "Fornecedor C"],
})
com_fornecedor = vendas.join(cadastro, on="produto", how="left", validate="m:1")
print(com_fornecedor.select("pedido_id", "produto", "fornecedor"))
assert com_fornecedor.height == vendas.height

shape: (8, 3)
┌───────────┬─────────┬──────────────┐
│ pedido_id ┆ produto ┆ fornecedor   │
│ ---       ┆ ---     ┆ ---          │
│ i64       ┆ str     ┆ str          │
╞═══════════╪═════════╪══════════════╡
│ 101       ┆ Caderno ┆ Fornecedor A │
│ 102       ┆ Caneta  ┆ Fornecedor B │
│ 103       ┆ Mochila ┆ Fornecedor C │
│ 104       ┆ Caderno ┆ Fornecedor A │
│ 105       ┆ Caneta  ┆ Fornecedor B │
│ 106       ┆ Mochila ┆ Fornecedor C │
│ 107       ┆ Caderno ┆ Fornecedor A │
│ 108       ┆ Caneta  ┆ Fornecedor B │
└───────────┴─────────┴──────────────┘


### 6.2 Concatenação de tabelas

Para juntar registros com o mesmo esquema, `pl.concat([...])` é uma alternativa ao `pd.concat([...])`.

In [12]:
parte_a = vendas.head(4)
parte_b = vendas.tail(4)
reunidas = pl.concat([parte_a, parte_b])
assert reunidas.height == 8
print(reunidas.select("pedido_id", "produto"))

shape: (8, 2)
┌───────────┬─────────┐
│ pedido_id ┆ produto │
│ ---       ┆ ---     │
│ i64       ┆ str     │
╞═══════════╪═════════╡
│ 101       ┆ Caderno │
│ 102       ┆ Caneta  │
│ 103       ┆ Mochila │
│ 104       ┆ Caderno │
│ 105       ┆ Caneta  │
│ 106       ┆ Mochila │
│ 107       ┆ Caderno │
│ 108       ┆ Caneta  │
└───────────┴─────────┘


## 7. Datas, janelas e formato da tabela

Polars possui operações temporais por meio de `.dt`. Para cálculos por grupo sem reduzir o número de linhas, use uma **window expression** com `.over(...)`.

In [13]:
tempo = vendas_calculadas.with_columns(
    ano=pl.col("data").dt.year(),
    mes=pl.col("data").dt.month(),
    mes_referencia=pl.col("data").dt.strftime("%Y-%m"),
)
com_participacao = tempo.with_columns(
    receita_media_regiao=pl.col("receita_liquida").mean().over("regiao").round(2)
)
print(com_participacao.select("pedido_id", "mes_referencia", "regiao", "receita_media_regiao"))

shape: (8, 4)
┌───────────┬────────────────┬────────┬──────────────────────┐
│ pedido_id ┆ mes_referencia ┆ regiao ┆ receita_media_regiao │
│ ---       ┆ ---            ┆ ---    ┆ ---                  │
│ i64       ┆ str            ┆ str    ┆ f64                  │
╞═══════════╪════════════════╪════════╪══════════════════════╡
│ 101       ┆ 2026-01        ┆ Sul    ┆ 56.5                 │
│ 102       ┆ 2026-01        ┆ Norte  ┆ 101.0                │
│ 103       ┆ 2026-01        ┆ Sul    ┆ 56.5                 │
│ 104       ┆ 2026-01        ┆ Norte  ┆ 101.0                │
│ 105       ┆ 2026-02        ┆ Sul    ┆ 56.5                 │
│ 106       ┆ 2026-02        ┆ Norte  ┆ 101.0                │
│ 107       ┆ 2026-02        ┆ Sul    ┆ 56.5                 │
│ 108       ┆ 2026-02        ┆ Norte  ┆ 101.0                │
└───────────┴────────────────┴────────┴──────────────────────┘


### 7.1 `pivot` e `unpivot`

- **Pivot:** transformar valores de uma categoria em colunas.
- **Unpivot:** transformar colunas em linhas.
- Para demonstração, agregamos antes para evitar ambiguidades sobre valores repetidos.

No Polars, `pivot` é uma operação eager; para planos lazy, reavalie a estratégia ou adie a operação até após `collect()`.

In [14]:
mensal_regiao = (
    vendas_calculadas.with_columns(pl.col("data").dt.strftime("%Y-%m").alias("mes"))
    .group_by("mes", "regiao")
    .agg(pl.col("receita_liquida").sum().round(2).alias("receita"))
)
painel = mensal_regiao.pivot(on="regiao", index="mes", values="receita").sort("mes")
volta_longa = painel.unpivot(index="mes", variable_name="regiao", value_name="receita")
print("Formato largo:\n", painel)
print("Formato longo:\n", volta_longa.sort("mes", "regiao"))

Formato largo:
 shape: (2, 3)
┌─────────┬───────┬───────┐
│ mes     ┆ Sul   ┆ Norte │
│ ---     ┆ ---   ┆ ---   │
│ str     ┆ f64   ┆ f64   │
╞═════════╪═══════╪═══════╡
│ 2026-01 ┆ 185.0 ┆ 95.0  │
│ 2026-02 ┆ 41.0  ┆ 309.0 │
└─────────┴───────┴───────┘
Formato longo:
 shape: (4, 3)
┌─────────┬────────┬─────────┐
│ mes     ┆ regiao ┆ receita │
│ ---     ┆ ---    ┆ ---     │
│ str     ┆ str    ┆ f64     │
╞═════════╪════════╪═════════╡
│ 2026-01 ┆ Norte  ┆ 95.0    │
│ 2026-01 ┆ Sul    ┆ 185.0   │
│ 2026-02 ┆ Norte  ┆ 309.0   │
│ 2026-02 ┆ Sul    ┆ 41.0    │
└─────────┴────────┴─────────┘


## 8. Arquivos: CSV e Parquet

Arquivos criados neste capítulo ficam em `outputs/`. CSV é conveniente para intercâmbio e inspeção; Parquet costuma preservar tipos de maneira mais adequada e é útil em fluxos analíticos. Não é preciso trazer datasets externos para executar estas células.

In [15]:
caminho_csv = SAIDAS / "polars_vendas_exemplo.csv"
caminho_parquet = SAIDAS / "polars_vendas_exemplo.parquet"
vendas_calculadas.write_csv(caminho_csv)
vendas_calculadas.write_parquet(caminho_parquet)
leitura_csv = pl.read_csv(caminho_csv, try_parse_dates=True)
leitura_parquet = pl.read_parquet(caminho_parquet)
print("CSV:", leitura_csv.shape, "| Parquet:", leitura_parquet.shape)
assert leitura_parquet.shape == vendas_calculadas.shape

CSV: (8, 11) | Parquet: (8, 11)


## 9. Lazy API: planejar, otimizar e só então executar

**Eager:** cada etapa calcula um DataFrame. **Lazy:** cada etapa compõe um `LazyFrame`; a consulta executa ao chamar `.collect()`.

Ao ler arquivos, `scan_csv` ou `scan_parquet` permite ao otimizador escolher melhor quais colunas e linhas precisam ser lidas. Dois conceitos importantes:

- **Projection pushdown:** ler apenas as colunas necessárias.
- **Predicate pushdown:** antecipar filtros para reduzir o volume lido.

`explain()` permite inspecionar o plano. Não interprete a saída como prova de que qualquer consulta lazy será sempre mais rápida; tamanho de dados, operações e ambiente influenciam.

In [16]:
consulta = (
    pl.scan_parquet(caminho_parquet)
    .filter(pl.col("regiao") == "Sul")
    .group_by("produto")
    .agg(pl.col("receita_liquida").sum().round(2).alias("receita_sul"))
    .sort("produto")
)
print("Tipo da consulta:", type(consulta).__name__)
print("Plano:\n", consulta.explain())
resultado_lazy = consulta.collect()
print("Resultado:\n", resultado_lazy)
assert isinstance(resultado_lazy, pl.DataFrame)

Tipo da consulta: LazyFrame
Plano:
 SORT BY [col("produto")]
  AGGREGATE[maintain_order: false]
    [col("receita_liquida").sum().round().alias("receita_sul")] BY [col("produto")]
    FROM
    simple π 2/2 ["produto", "receita_liquida"]
      Parquet SCAN [outputs/polars_vendas_exemplo.parquet]
      PROJECT 3/11 COLUMNS
      SELECTION: col("regiao") == "Sul"
      ESTIMATED ROWS: 8
Resultado:
 shape: (3, 2)
┌─────────┬─────────────┐
│ produto ┆ receita_sul │
│ ---     ┆ ---         │
│ str     ┆ f64         │
╞═════════╪═════════════╡
│ Caderno ┆ 75.0        │
│ Caneta  ┆ 16.0        │
│ Mochila ┆ 135.0       │
└─────────┴─────────────┘


### 9.1 Começando de um DataFrame existente

O método `.lazy()` cria um plano sobre dados já carregados. Ele não é equivalente a `scan_*` quanto à possibilidade de evitar a leitura inicial do arquivo.

In [17]:
lazy_memoria = (
    vendas.lazy()
    .filter(pl.col("quantidade") > 2)
    .select("produto", "quantidade")
)
print(lazy_memoria.collect())

shape: (4, 2)
┌─────────┬────────────┐
│ produto ┆ quantidade │
│ ---     ┆ ---        │
│ str     ┆ i64        │
╞═════════╪════════════╡
│ Caneta  ┆ 5          │
│ Caderno ┆ 3          │
│ Caneta  ┆ 4          │
│ Caneta  ┆ 6          │
└─────────┴────────────┘


## 10. Comparação direta com Pandas

As duas bibliotecas são úteis. Pandas tem ecossistema amplo e familiar; Polars privilegia expressões e dispõe de consulta lazy. **Não conclua que uma substitui a outra em qualquer projeto.** Para comparar, use exatamente o mesmo conjunto de dados e confira os resultados.

In [18]:
pandas_vendas = pd.DataFrame(vendas.to_dict(as_series=False))

# Pandas: filtrar + agrupar
resumo_pd = (
    pandas_vendas.loc[pandas_vendas["quantidade"] >= 2]
    .groupby("regiao", as_index=False)["quantidade"].sum()
    .sort_values("regiao")
    .reset_index(drop=True)
)
# Polars: a mesma lógica com expressões
resumo_pl = (
    vendas.filter(pl.col("quantidade") >= 2)
    .group_by("regiao")
    .agg(pl.col("quantidade").sum())
    .sort("regiao")
)
print("Pandas:\n", resumo_pd)
print("Polars:\n", resumo_pl)
assert resumo_pd["quantidade"].tolist() == resumo_pl["quantidade"].to_list()

Pandas:
   regiao  quantidade
0  Norte          16
1    Sul           6
Polars:
 shape: (2, 2)
┌────────┬────────────┐
│ regiao ┆ quantidade │
│ ---    ┆ ---        │
│ str    ┆ i64        │
╞════════╪════════════╡
│ Norte  ┆ 16         │
│ Sul    ┆ 6          │
└────────┴────────────┘


### 10.1 Conversão entre ecossistemas

`pl.from_pandas` e `DataFrame.to_pandas()` podem exigir **PyArrow**, dependendo do caminho de conversão e do ambiente. Para manter este notebook instalável com poucas dependências, as comparações acima usam um dicionário Python comum. Em projetos reais, avalie tipos, nulos, datas e custo da conversão antes de transferir DataFrames grandes.

Outras diferenças importantes: o Polars não utiliza índice de rótulos, alguns resultados agrupados não têm ordem garantida, `null` difere de `NaN`, e o comportamento de conversões pode ser mais estrito.

## 11. Estudo de caso integrado — relatório mensal de vendas

**Pergunta analítica:** qual região apresentou maior receita líquida, qual produto vendeu mais unidades e como a receita variou por mês?

**Passos:** validar dados, tratar descontos ausentes com regra documentada, calcular métricas, agregar, ordenar e exportar. Um pipeline bem construído separa as transformações da interpretação.

In [19]:
# Validação mínima do contrato de dados
campos_obrigatorios = {"pedido_id", "data", "produto", "regiao", "quantidade", "preco_unitario"}
assert campos_obrigatorios.issubset(set(vendas.columns))
assert vendas.select((pl.col("quantidade") > 0).all()).item()
assert vendas.select((pl.col("preco_unitario") >= 0).all()).item()
assert vendas["pedido_id"].n_unique() == vendas.height

# Pipeline analítico (sem alterar `vendas`)
relatorio_base = (
    vendas.lazy()
    .with_columns(pl.col("desconto_pct").fill_null(0).alias("desconto_tratado"))
    .with_columns(
        (pl.col("quantidade") * pl.col("preco_unitario") *
         (1 - pl.col("desconto_tratado"))).round(2).alias("receita_liquida"),
        pl.col("data").dt.strftime("%Y-%m").alias("mes"),
    )
    .collect()
)
receita_regiao = relatorio_base.group_by("regiao").agg(
    pl.col("receita_liquida").sum().round(2).alias("receita")
).sort("receita", descending=True)
quantidade_produto = relatorio_base.group_by("produto").agg(
    pl.col("quantidade").sum().alias("unidades")
).sort("unidades", descending=True)
receita_mensal = relatorio_base.group_by("mes").agg(
    pl.col("receita_liquida").sum().round(2).alias("receita")
).sort("mes")
print("Receita por região:\n", receita_regiao)
print("Unidades por produto:\n", quantidade_produto)
print("Receita por mês:\n", receita_mensal)

Receita por região:
 shape: (2, 2)
┌────────┬─────────┐
│ regiao ┆ receita │
│ ---    ┆ ---     │
│ str    ┆ f64     │
╞════════╪═════════╡
│ Norte  ┆ 404.0   │
│ Sul    ┆ 226.0   │
└────────┴─────────┘
Unidades por produto:
 shape: (3, 2)
┌─────────┬──────────┐
│ produto ┆ unidades │
│ ---     ┆ ---      │
│ str     ┆ i64      │
╞═════════╪══════════╡
│ Caneta  ┆ 15       │
│ Caderno ┆ 6        │
│ Mochila ┆ 3        │
└─────────┴──────────┘
Receita por mês:
 shape: (2, 2)
┌─────────┬─────────┐
│ mes     ┆ receita │
│ ---     ┆ ---     │
│ str     ┆ f64     │
╞═════════╪═════════╡
│ 2026-01 ┆ 280.0   │
│ 2026-02 ┆ 350.0   │
└─────────┴─────────┘


In [20]:
receita_regiao.write_csv(SAIDAS / "polars_receita_por_regiao.csv")
quantidade_produto.write_csv(SAIDAS / "polars_unidades_por_produto.csv")
receita_mensal.write_csv(SAIDAS / "polars_receita_mensal.csv")

regiao_lider = receita_regiao["regiao"][0]
produto_lider = quantidade_produto["produto"][0]
print("Região líder em receita:", regiao_lider)
print("Produto líder em unidades:", produto_lider)
print("Arquivos de relatório salvos em outputs/.")

Região líder em receita: Norte
Produto líder em unidades: Caneta
Arquivos de relatório salvos em outputs/.


### 11.1 Interpretação responsável

Os resultados descrevem **somente as oito linhas fictícias**. Eles não justificam conclusões sobre consumidores reais. Antes de aplicar esse fluxo a uma base real, confira a unidade de cada linha (item versus pedido), possíveis pedidos com vários itens, moeda, impostos, descontos, valores estornados e qualidade das chaves. Neste caso, o identificador é único por linha; por isso o número de linhas coincide com o de pedidos fictícios.

## 12. Exercícios práticos (tente antes de abrir a solução)

**Exercício A — filtros e seleção:** encontre os produtos vendidos na região Norte com quantidade superior a três. Retorne apenas produto e quantidade, ordenados pela quantidade.

**Exercício B — métricas:** calcule o faturamento bruto (`quantidade × preço_unitario`) para cada linha e a soma total.

**Exercício C — grupos:** calcule o total de unidades por produto, em ordem decrescente.

**Exercício D — qualidade:** conte quantos descontos estão ausentes; substitua-os por zero em uma nova tabela, sem sobrescrever a original.

**Exercício E — lazy:** leia o Parquet salvo no capítulo 8 de forma lazy, filtre quantidade maior ou igual a 3 e obtenha o total por região. Execute com `.collect()`.

**Exercício F — janela:** acrescente a cada registro a quantidade total vendida de seu produto sem reduzir a quantidade de linhas.

**Desafio integrador:** a partir de `vendas`, gere um CSV com receita líquida por categoria e mês. Nome sugerido: `outputs/polars_receita_categoria_mes.csv`.

### Soluções comentadas

As células a seguir mostram uma forma de resolver cada problema. Compare com sua tentativa: a mesma resposta pode ser obtida por encadeamentos diferentes.

In [21]:
# Exercício A: expressão booleana + projeção
sol_a = (vendas.filter((pl.col("regiao") == "Norte") & (pl.col("quantidade") > 3))
         .select("produto", "quantidade").sort("quantidade"))
print(sol_a)

shape: (2, 2)
┌─────────┬────────────┐
│ produto ┆ quantidade │
│ ---     ┆ ---        │
│ str     ┆ i64        │
╞═════════╪════════════╡
│ Caneta  ┆ 5          │
│ Caneta  ┆ 6          │
└─────────┴────────────┘


In [22]:
# Exercício B: expressão calculada e redução
sol_b = vendas.with_columns(
    (pl.col("quantidade") * pl.col("preco_unitario")).alias("faturamento_bruto")
)
print(sol_b.select("pedido_id", "faturamento_bruto"))
print("Total:", sol_b["faturamento_bruto"].sum())

shape: (8, 2)
┌───────────┬───────────────────┐
│ pedido_id ┆ faturamento_bruto │
│ ---       ┆ ---               │
│ i64       ┆ f64               │
╞═══════════╪═══════════════════╡
│ 101       ┆ 50.0              │
│ 102       ┆ 20.0              │
│ 103       ┆ 150.0             │
│ 104       ┆ 75.0              │
│ 105       ┆ 16.0              │
│ 106       ┆ 300.0             │
│ 107       ┆ 25.0              │
│ 108       ┆ 24.0              │
└───────────┴───────────────────┘
Total: 660.0


In [23]:
# Exercício C: agregado nomeado
sol_c = (vendas.group_by("produto")
         .agg(pl.col("quantidade").sum().alias("total_unidades"))
         .sort("total_unidades", descending=True))
print(sol_c)

shape: (3, 2)
┌─────────┬────────────────┐
│ produto ┆ total_unidades │
│ ---     ┆ ---            │
│ str     ┆ i64            │
╞═════════╪════════════════╡
│ Caneta  ┆ 15             │
│ Caderno ┆ 6              │
│ Mochila ┆ 3              │
└─────────┴────────────────┘


In [24]:
# Exercício D: decisão de imputação explícita
faltantes = vendas.select(pl.col("desconto_pct").is_null().sum()).item()
sol_d = vendas.with_columns(pl.col("desconto_pct").fill_null(0.0))
print("Descontos ausentes originalmente:", faltantes)
print("Após imputação:", sol_d["desconto_pct"].null_count())
assert vendas["desconto_pct"].null_count() == 1

Descontos ausentes originalmente: 1
Após imputação: 0


In [25]:
# Exercício E: primeiro construímos o plano, depois coletamos
sol_e = (pl.scan_parquet(caminho_parquet)
         .filter(pl.col("quantidade") >= 3)
         .group_by("regiao")
         .agg(pl.col("quantidade").sum().alias("total_unidades"))
         .sort("regiao")
         .collect())
print(sol_e)

shape: (2, 2)
┌────────┬────────────────┐
│ regiao ┆ total_unidades │
│ ---    ┆ ---            │
│ str    ┆ i64            │
╞════════╪════════════════╡
│ Norte  ┆ 14             │
│ Sul    ┆ 4              │
└────────┴────────────────┘


In [26]:
# Exercício F: over retorna um valor de grupo por linha
sol_f = vendas.with_columns(
    pl.col("quantidade").sum().over("produto").alias("unidades_do_produto")
)
print(sol_f.select("pedido_id", "produto", "quantidade", "unidades_do_produto"))
assert sol_f.height == vendas.height

shape: (8, 4)
┌───────────┬─────────┬────────────┬─────────────────────┐
│ pedido_id ┆ produto ┆ quantidade ┆ unidades_do_produto │
│ ---       ┆ ---     ┆ ---        ┆ ---                 │
│ i64       ┆ str     ┆ i64        ┆ i64                 │
╞═══════════╪═════════╪════════════╪═════════════════════╡
│ 101       ┆ Caderno ┆ 2          ┆ 6                   │
│ 102       ┆ Caneta  ┆ 5          ┆ 15                  │
│ 103       ┆ Mochila ┆ 1          ┆ 3                   │
│ 104       ┆ Caderno ┆ 3          ┆ 6                   │
│ 105       ┆ Caneta  ┆ 4          ┆ 15                  │
│ 106       ┆ Mochila ┆ 2          ┆ 3                   │
│ 107       ┆ Caderno ┆ 1          ┆ 6                   │
│ 108       ┆ Caneta  ┆ 6          ┆ 15                  │
└───────────┴─────────┴────────────┴─────────────────────┘


In [27]:
# Desafio integrador: limpeza + cálculo + agrupamento + exportação
sol_integrador = (
    vendas.with_columns(pl.col("desconto_pct").fill_null(0.0).alias("desconto_ajustado"))
    .with_columns(
        mes=pl.col("data").dt.strftime("%Y-%m"),
        receita_liquida=(pl.col("quantidade") * pl.col("preco_unitario") *
                         (1 - pl.col("desconto_ajustado"))).round(2),
    )
    .group_by("categoria", "mes")
    .agg(pl.col("receita_liquida").sum().round(2).alias("receita"))
    .sort("mes", "categoria")
)
sol_integrador.write_csv(SAIDAS / "polars_receita_categoria_mes.csv")
print(sol_integrador)

shape: (4, 3)
┌────────────┬─────────┬─────────┐
│ categoria  ┆ mes     ┆ receita │
│ ---        ┆ ---     ┆ ---     │
│ str        ┆ str     ┆ f64     │
╞════════════╪═════════╪═════════╡
│ Acessórios ┆ 2026-01 ┆ 135.0   │
│ Papelaria  ┆ 2026-01 ┆ 145.0   │
│ Acessórios ┆ 2026-02 ┆ 285.0   │
│ Papelaria  ┆ 2026-02 ┆ 65.0    │
└────────────┴─────────┴─────────┘


## 12.1 Contratos de dados e validação explícita

Ao importar CSV/Parquet, verifique nomes, tipos, unicidade de chaves, limites e unidades. A inferência automática pode variar conforme os dados; sempre que necessário, forneça `schema_overrides` e valide com `assert` ou testes formais. **Importante:** os exemplos anteriores são dados fictícios com apenas oito transações; não permitem comparar desempenho de bibliotecas de maneira justa.

In [28]:
# Contrato mínimo e verificação de tipos, preservando a tabela de origem.
assert vendas.schema['pedido_id'] == pl.Int64
assert vendas.schema['quantidade'] == pl.Int64
assert vendas['pedido_id'].n_unique() == vendas.height
assert vendas.select((pl.col('quantidade') > 0).all()).item()
print('Contrato de dados verificado.')

Contrato de dados verificado.


## 12.2 Plano de consulta, pushdown e otimização

Na execução *lazy*, o Polars pode projetar apenas colunas necessárias (**projection pushdown**) e antecipar filtros (**predicate pushdown**) na leitura de fontes compatíveis. Isso reduz transferência e transformação de dados, mas a otimização concreta depende do formato e do plano. Use `.explain()` para inspecionar o planejamento; `.collect()` executa a consulta. Não generalize sobre velocidade sem medições repetidas, aquecimento e controle de hardware e dados.

In [29]:
plano_estudo = (pl.scan_parquet(caminho_parquet)
                 .filter(pl.col('quantidade') >= 2)
                 .select('produto', 'quantidade'))
explicacao = plano_estudo.explain()
assert isinstance(explicacao, str) and explicacao
print(explicacao[:1200])
print(plano_estudo.collect())

Parquet SCAN [outputs/polars_vendas_exemplo.parquet]
PROJECT 2/11 COLUMNS
SELECTION: col("quantidade") >= 2
ESTIMATED ROWS: 8
shape: (6, 2)
┌─────────┬────────────┐
│ produto ┆ quantidade │
│ ---     ┆ ---        │
│ str     ┆ i64        │
╞═════════╪════════════╡
│ Caderno ┆ 2          │
│ Caneta  ┆ 5          │
│ Caderno ┆ 3          │
│ Caneta  ┆ 4          │
│ Mochila ┆ 2          │
│ Caneta  ┆ 6          │
└─────────┴────────────┘


## 12.3 Exercício avançado — duplicidades em tabelas relacionadas

**Desafio:** verifique a unicidade da chave de um cadastro antes de uma junção. Por que duplicatas podem multiplicar registros após `join`?

**Solução:** compare `n_unique()` ao número de linhas e interrompa o fluxo quando a cardinalidade esperada não for respeitada.

In [30]:
# Gabarito: validação de cardinalidade antes de uma junção.
assert cadastro['produto'].n_unique() == cadastro.height
resultado_join = vendas.join(cadastro, on='produto', how='left')
assert resultado_join.height == vendas.height
print('Junção sem multiplicação inesperada de linhas.')

Junção sem multiplicação inesperada de linhas.


## 12.4 Checklist de análise responsável

- Documentar a fonte, unidade de observação e critérios de transformação.
- Conferir tipos e valores nulos antes de agregar.
- Investigar duplicatas das chaves de junção.
- Distinguir resultados derivados de amostras fictícias e evidências de produção.
- Registrar versão do Polars e formato dos arquivos.
- Medir desempenho com datasets adequados, diversas repetições e condições controladas.
- Escrever resultados apenas em `outputs/`, sem sobrescrever entradas originais.

## 13. Consulta Rápida — Polars

Use esta seção como lembrete depois de estudar os capítulos. Os comandos supõem `import polars as pl` e um `DataFrame` chamado `df`.

| Necessidade | Polars | Observação |
|---|---|---|
| Criar DataFrame | `pl.DataFrame({"a": [1, 2]})` | Colunas tipadas |
| Ver colunas/tipos | `df.columns`, `df.schema` | Inspeção inicial |
| Primeiras linhas | `df.head(5)` | Não altera `df` |
| Selecionar | `df.select("a")` | Escolhe colunas |
| Filtrar | `df.filter(pl.col("a") > 1)` | Escolhe linhas |
| Nova coluna | `df.with_columns((pl.col("a") * 2).alias("dobro"))` | Retorna novo DataFrame |
| Ordenar | `df.sort("a", descending=True)` | Ordem explícita |
| Agregar | `df.group_by("grupo").agg(pl.col("a").sum())` | Uma linha por grupo |
| Janela | `pl.col("a").sum().over("grupo")` | Preserva linhas |
| Nulos | `df.null_count()` | `NaN` é diferente de `null` |
| Preencher nulos | `pl.col("a").fill_null(0)` | Só quando justificável |
| Converter tipo | `pl.col("a").cast(pl.Int64)` | `strict=False` permite nulo em falha |
| Unir tabelas | `df.join(outro, on="id", how="left")` | Verificar cardinalidade |
| Empilhar linhas | `pl.concat([df1, df2])` | Esquemas compatíveis |
| Data → ano | `pl.col("data").dt.year()` | Data tipada |
| Ler CSV | `pl.read_csv("arquivo.csv")` | Eager |
| Ler CSV lazy | `pl.scan_csv("arquivo.csv")` | Retorna LazyFrame |
| Parquet | `pl.read_parquet("arquivo.parquet")` | Preserva tipos |
| Salvar CSV | `df.write_csv("saida.csv")` | Produz arquivo |
| Tornar lazy | `df.lazy()` | Ainda não executa consultas |
| Executar lazy | `consulta.collect()` | Retorna DataFrame |
| Ver plano lazy | `consulta.explain()` | Ajuda a entender otimizações |

### Pandas ↔ Polars: mapa mental

| Pandas | Polars |
|---|---|
| `df[["a", "b"]]` | `df.select("a", "b")` |
| `df[df["a"] > 1]` | `df.filter(pl.col("a") > 1)` |
| `df.assign(nova=...)` | `df.with_columns(...)` |
| `df.groupby("x").agg(...)` | `df.group_by("x").agg(...)` |
| `pd.read_csv(...)` | `pl.read_csv(...)` |
| `df.to_csv(...)` | `df.write_csv(...)` |
| `df.isna().sum()` | `df.null_count()` (ver `NaN`) |
| `df.sort_values("a")` | `df.sort("a")` |

**Erros frequentes:** esquecer `.collect()`; supor um índice estilo Pandas; usar `and` dentro de expressões; esquecer parênteses em filtros; supor ordem automática após agrupamento; confundir `null` com `NaN`; assumir que colunas calculadas no mesmo `with_columns` já estão disponíveis em outras expressões daquele mesmo contexto.

### Para aprofundar — documentação oficial

- [Guia de migração do Pandas](https://docs.pola.rs/user-guide/migration/pandas/)
- [Expressões e contextos](https://docs.pola.rs/user-guide/concepts/expressions-and-contexts/)
- [Lazy API](https://docs.pola.rs/user-guide/concepts/lazy-api/)
- [Tipos e estruturas](https://docs.pola.rs/user-guide/concepts/data-types-and-structures/)

**Próximo estudo opcional:** aplicar o mesmo pipeline a um CSV real pequeno, documentando a origem, a qualidade dos dados e cada regra de transformação.